# 📊 Social Media Sentiment Analysis – Exploratory Data Analysis

This notebook walks through the full pipeline step by step:
1. Dataset generation & exploration
2. Text cleaning
3. NLP preprocessing
4. TF-IDF feature extraction
5. Model training & evaluation
6. Visualization


In [ ]:
# Install dependencies (run once)
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Libraries loaded successfully')

## Step 1: Generate & Load Dataset

In [ ]:
from data.generate_dataset import generate_dataset

# Generate 1500 synthetic social media posts
df = generate_dataset(n=1500, output_path='../data/social_media_data.csv')
print(f'\nDataset shape: {df.shape}')
df.head(10)

In [ ]:
# Basic info
print('Dataset Info:')
print(df.info())
print('\nSentiment Distribution:')
print(df['sentiment'].value_counts())
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Sentiment distribution chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {'Positive': '#4CAF50', 'Negative': '#F44336', 'Neutral': '#2196F3'}
counts = df['sentiment'].value_counts()

axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=[colors[l] for l in counts.index], startangle=90)
axes[0].set_title('Sentiment Distribution (Pie)', fontweight='bold')

axes[1].bar(counts.index, counts.values,
            color=[colors[l] for l in counts.index])
axes[1].set_title('Sentiment Distribution (Bar)', fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../images/sentiment_distribution_nb.png', dpi=150)
plt.show()
print('✅ Chart saved')

## Step 2: Text Cleaning

In [ ]:
from src.clean_text import clean_text

samples = [
    'Zomato delivery was super fast today! Loved it 🎉 #Zomato @zomato_support http://zomato.com',
    'Netflix keeps buffering 😡!! Very frustrating experience.',
    'Placed my Amazon order. Waiting for delivery. #Amazon2024',
]

print('TEXT CLEANING DEMO\n' + '='*60)
for s in samples:
    print(f'Original : {s}')
    print(f'Cleaned  : {clean_text(s)}')
    print('-'*60)

## Step 3: NLP Preprocessing

In [ ]:
from src.preprocess import preprocess_text, preprocess_dataframe

for s in samples:
    print(f'Original  : {s}')
    print(f'Processed : {preprocess_text(s)}')
    print('-'*60)

# Apply to full dataset
df = preprocess_dataframe(df)
print('\nPreprocessing applied to full dataset')
df[['text', 'cleaned_text', 'processed_text']].head(5)

## Step 4: TF-IDF Feature Extraction

In [ ]:
from src.feature_extraction import fit_transform, save_vectorizer

X, vectorizer = fit_transform(df['processed_text'])
print(f'TF-IDF Matrix shape: {X.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print(f'Top 20 features: {vectorizer.get_feature_names_out()[:20].tolist()}')

save_vectorizer(vectorizer, '../models/tfidf_vectorizer.pkl')

## Step 5: Model Training & Evaluation

In [ ]:
import os
os.chdir('..')  # move to project root

from src.train_model import run_training
results = run_training()

## Step 6: Live Predictions Demo

In [ ]:
from src.predict import predict_vader, predict_ml, load_artifacts

test_texts = [
    'Zomato delivery was super fast today! Loved the packaging!',
    'Netflix keeps buffering. Very frustrating experience!',
    'Placed my Zomato order. Waiting for delivery.',
    'Amazon delivered a damaged product. Refund is a nightmare!',
]

print('VADER Predictions:')
for r in predict_vader(test_texts):
    e = {'Positive':'✅','Negative':'❌','Neutral':'⚪'}[r['sentiment']]
    print(f"{e} [{r['sentiment']:8s}] → {r['text'][:50]}")

print('\nML Predictions:')
model, le, vectorizer = load_artifacts()
for r in predict_ml(test_texts, model, le, vectorizer):
    e = {'Positive':'✅','Negative':'❌','Neutral':'⚪'}[r['sentiment']]
    print(f"{e} [{r['sentiment']:8s}] (conf={r['confidence']}%) → {r['text'][:50]}")

## ✅ Notebook Complete!

Next step: Launch the interactive dashboard
```bash
streamlit run app/dashboard.py
```